# 01 — Explore LandIQ shapefiles

**Goal:** See layers, CRS, attribute columns, and crop codes so you can identify tomato.

1. Copy LandIQ files into `data/raw/landiq/` (see `data/README.md`).
2. Run the cells below and note which column holds crop type and the exact values for tomato.
3. Copy `configs/paths.example.yaml` to `configs/paths.local.yaml` and set `landiq.crop_column` and `landiq.tomato_values`.

In [ ]:
from pathlib import Path

import geopandas as gpd

from src.landiq.inspect import load_shapefile, summarize_gdf, value_counts_for_columns

RAW = Path("../data/raw/landiq")
# LandIQ is usually organized as raw/landiq/<year>/… with zips — extract zips first, then .shp appear.
shps = sorted(RAW.rglob("*.shp"))
if not shps:
    raise FileNotFoundError(
        f"No .shp under {RAW.resolve()}. Extract each year's *_crop_mapping_*.zip into that year folder."
    )
# Pick one year to inspect, or set to 0 to use the first file found
YEAR_FOLDER = "2024"  # change e.g. to "2020"
candidates = [p for p in shps if YEAR_FOLDER in p.parts]
shp_path = candidates[0] if candidates else shps[0]
print("Using:", shp_path)

gdf = load_shapefile(shp_path)
print(summarize_gdf(gdf))

In [ ]:
# List string/object columns likely to hold crop labels
candidates = [c for c in gdf.columns if gdf[c].dtype == object or "crop" in c.lower()]
print("Candidate columns:", candidates)
vc = value_counts_for_columns(gdf, candidates[:8], top_n=40)
for col, s in vc.items():
    print("\n", col, "\n", s)